# SetFit contrastive baseline - StereoQueerEval Task A

Dedicated notebook for the **non-prompt control**: contrastive fine-tuning of an
embedding backbone (`run_setfit.py`) followed by a small classifier head. Same
video-grouped split, seeds and metric (validation macro-F1) as the prompt-MLM
pipeline in `train_mmbert.ipynb`, but the text is fed WITHOUT the
question/answer `[MASK]` framing, so the comparison isolates whether the prompt
vehicle adds value.

It walks a **body-training ladder** on the SAME data and split, cheapest first:

| stage | flag | what the contrastive phase trains |
|-------|------|-----------------------------------|
| `headonly` | `--head-only` | nothing (head on fixed *pretrained* embeddings) |
| `tail2`    | `--unfreeze-body-layers 2` | last 2 encoder blocks + pooling |
| `tail4`    | `--unfreeze-body-layers 4` | last 4 encoder blocks + pooling |
| `full`     | (default) | the whole body (stock SetFit) |

Then it runs a `--head-sweep` on the trained body and prints one combined
results table + bar chart (its own stages, plus the prompt-MLM `control` runs
from `train_mmbert.ipynb` if their histories are present).

Backbone: `vllm-sr/mmbert-embed-32k-2d-matryoshka` (mmBERT-Embed, a fine-tune of
`jhu-clsp/mmbert-base`; falls back to multilingual MiniLM on load failure). On a
Kaggle T4 the weights ship in bf16, which a T4 cannot compute, so default is fp16
(`--no-amp` forces fp32).

Read the cost of the contrastive phase: `--contrastive-batch-size` counts PAIRS
(2 x `max-seq-length` sequences each). `8` keeps a 307M backbone inside a 16 GB
T4; the frozen/partial stages train fewer params, so they may tolerate more.


## 1. Setup

Clones `src/` (or fast-forwards it) and installs the pinned requirements. The
SetFit stack (`setfit==1.2.0`, `sentence-transformers==6.1.0`, ...) is pinned on
purpose: `run_setfit.py` uses SetFit internals and runs an API self-check.


In [ ]:
import os
import subprocess, sys

if not os.path.isdir('Try'):
    subprocess.run('git clone --depth 1 https://github.com/AmnO-O/Try.git', shell=True, check=True)
else:
    subprocess.run('git -C Try pull --ff-only', shell=True, check=True)
os.chdir('Try')
os.makedirs('data', exist_ok=True)
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('kernel :', 'Kaggle' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'Colab' if 'COLAB_GPU' in os.environ else 'local')
print('repo   :', os.getcwd())
!ls src


## 1. Locate the TSVs

Downloads are gitignored, so the clone has no data. This cell searches the usual
places -- your Kaggle dataset path, Colab `/content`, and a local `data/` -- and
sets `LGBT_DATA_DIR` to the directory that actually contains the files.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.path.join(os.environ['KAGGLE_INPUT_PATH']))
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    hits = []
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return os.path.dirname(hits[0]), hits
    return None, hits

data_dir, found = _first(candidates)
if not found:
    raise RuntimeError('No StereoQueerEval_*_training.tsv found. Add your Kaggle dataset and re-run.')

os.environ['LGBT_DATA_DIR'] = data_dir
os.environ['LGBT_WEIGHTS_DIR'] = os.path.join(os.getcwd(), 'weights')
print('data dir :', data_dir)
print('files    :', [os.path.basename(p) for p in found])
assert len(found) == 3, f'expect EN+IT+NL, got {len(found)}'


## 2. Sanity check the data

Row counts, label balance, video-grouped split -- cheap checks before any GPU time.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

df = data_mod.load_frame(max_len=MAX_LEN)
print('rows:', len(df), '| labels:', data_mod.label_distribution(df))
print('by lang:', df['lang'].value_counts().to_dict())
print('longest prompt words:', int(df['prompt'].str.split().str.len().max()))

tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))


## 3. Run config

`SEEDS = (40,)` first to verify the full GPU path end-to-end, then widen to
`(40, 41, 42)`. Each stage writes `weights/setfit_s<seed>_<stage>/` plus
`..._history.csv` (real per-epoch log) and `..._run.json` (full metadata,
including `body_unfreeze_layers`).

`--contrastive-batch-size 8` keeps the 307M backbone inside a 16 GB T4. Stages
**B** train only a few blocks (+pooling), so they can afford a larger batch if
you have headroom; keep it at 8 for a like-for-like comparison. The head phase
(Design A) always pre-encodes the frozen body once and never trains it.


In [ ]:
import os
import subprocess
import pandas as pd

# Pinned SetFit stack (requirements.txt also pins it; re-assert for Kaggle images).
subprocess.run('pip install -q setfit==1.2.0 sentence-transformers==6.1.0 transformers==5.19.0 datasets==5.1.0 evaluate==0.4.6 scikit-learn==1.9.0', shell=True, check=True)

SETFIT_BACKBONE = 'vllm-sr/mmbert-embed-32k-2d-matryoshka'
SEEDS = (40,)   # widen to (40, 41, 42) once the ladder looks sane

# body-training mode per stage (extra run_setfit.py flags)
STAGES = {
    'headonly': ['--head-only'],
    'tail2':    ['--unfreeze-body-layers', '2'],
    'tail4':    ['--unfreeze-body-layers', '4'],
    'full':     [],
}


def run_stage(stage, seed, batch=8):
    out = f'weights/setfit_s{seed}_{stage}'
    cmd = ['python', 'src/run_setfit.py', '--languages', 'EN', 'IT', 'NL',
           '--split-seed', str(seed), '--seed', str(seed),
           '--contrastive-epochs', '1', '--num-iterations', '5',
           '--head-epochs', '20', '--patience', '5',
           '--contrastive-batch-size', str(batch),
           '--backbone', SETFIT_BACKBONE,
           *STAGES[stage], '--out', out]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=os.getcwd())
    return out


print('backbone:', SETFIT_BACKBONE)
print('seeds   :', SEEDS)
print('stages  :', list(STAGES))


## 4. Stage A -- frozen body (`--head-only`)

No contrastive phase: a torch head is fit on **fixed pretrained embeddings**.
This is the cheapest control and the floor the trained-body stages must beat.
~1 min/seed.


In [ ]:
for s in SEEDS:
    run_stage('headonly', s)


## 5. Stage B -- partial body, last 2 blocks (`--unfreeze-body-layers 2`)

Contrastive phase trains only the last 2 encoder blocks + pooling (~3% of the
307M body): far less activation and optimizer memory, so it fits a larger batch
on a T4. The question is whether this matches full-body training on val macro-F1.


In [ ]:
for seed in SEEDS:
    run_stage('tail2', seed)   # bump batch=... if you have headroom


## 5. Stage B' -- partial body (`--unfreeze-body-layers 4`)

Same idea, more capacity. Optional -- skip if `tail2` already matches `full`.


In [ ]:
for seed in SEEDS:
    run_stage('tail4', seed)


## 5. Stage C -- full body (stock SetFit)

The whole body is contrastively fine-tuned. This is the canonical SetFit control;
expect ~2-2.5 h/seed on a T4 at `--contrastive-batch-size 8`. Keep this cell last
so the cheap stages (A/B) can be checked first.


In [ ]:
for seed in SEEDS:
    run_stage('full', seed)


## 5. Head sweep on the trained body

Reuses the **full-body** output via `--body-from` and re-fits several heads on the
identical embeddings (the contrastive phase runs once). One comparison table:

- `torch` -- single `Linear(emb, 2)` (SetFit's default head)
- `mlp`   -- `Linear(emb, 256) -> GELU -> Dropout(0.2) -> Linear(256, 2)`
- `lr:C`  -- sklearn `LogisticRegression` at inverse-regularization `C`

Differences are usually within seed noise; this is a cheap A/B, not a lever.


In [ ]:
seed = SEEDS[0]
body = f'weights/setfit_s{seed}_full'
assert os.path.isdir(body), f'missing body {body} - run the Full-body cell first'

cmd = ['python', 'src/run_setfit.py', '--languages', 'EN', 'IT', 'NL',
       '--split-seed', str(seed), '--seed', str(seed),
       '--head-sweep', '--sweep-heads', 'torch,mlp,lr:0.1,lr:1,lr:10',
       '--backbone', SETFIT_BACKBONE, '--body-from', body, '--out', body]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 6. Results -- body-training ladder

Best-epoch validation macro-F1 per stage (one point per seed). The `prompt_mlm`
row appears only if `train_mmbert.ipynb` wrote `weights/control_s<seed>_history.csv`
in this session. Differences of <~0.01-0.02 are within seed noise.


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rows = []
for stage in STAGES:
    for seed in SEEDS:
        hist = f'weights/setfit_s{seed}_{stage}_history.csv'
        if not os.path.exists(hist):
            continue
        h = pd.read_csv(hist)
        best = h.loc[h['val_macro_f1'].idxmax()]
        rows.append({'stage': stage, 'seed': seed,
                     'val_macro_f1': float(best['val_macro_f1']),
                     'val_acc': float(best['val_acc']),
                     'best_epoch': int(best['epoch'])})

for seed in SEEDS:  # prompt-MLM control from train_mmbert.ipynb, if present
    hist = f'weights/control_s{seed}_history.csv'
    if os.path.exists(hist):
        h = pd.read_csv(hist)
        best = h.loc[h['val_macro_f1'].idxmax()]
        rows.append({'stage': 'prompt_mlm', 'seed': seed,
                     'val_macro_f1': float(best['val_macro_f1']),
                     'val_acc': float(best['val_acc']),
                     'best_epoch': int(best['epoch'])})

if not rows:
    print('No histories yet - run at least one stage above.')
else:
    df = pd.DataFrame(rows)
    order = ['headonly', 'tail2', 'tail4', 'full', 'prompt_mlm']
    order = [s for s in order if s in set(df['stage'])]

    # comparison matrix: rows = stage, cols = seed, values = best val macro-F1
    matrix = df.pivot_table(index='stage', columns='seed', values='val_macro_f1',
                            aggfunc='max').reindex(order)
    matrix['mean'] = matrix.mean(axis=1).round(4)
    matrix['best_epoch'] = (df.groupby('stage')['best_epoch'].max().reindex(order))
    print('=== best val macro-F1 matrix (rows: stage, cols: seed) ===')
    print(matrix.round(4).to_string())

    ax = matrix.drop(columns=['mean', 'best_epoch']).plot.bar(rot=0, figsize=(9, 4.5))
    ax.set_ylabel('best val macro-F1')
    ax.set_title('SetFit body-training ladder vs prompt-MLM')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## 7. Per-language check

`*_run.json` holds `per_language` metrics (EN vs IT/NL). Small training sets can
hide per-language regressions behind a good macro-F1, so inspect it before
writing anything up.


In [ ]:
import json

for stage in STAGES:
    for seed in SEEDS:
        meta = f'weights/setfit_s{seed}_{stage}_run.json'
        if not os.path.exists(meta):
            continue
        m = json.load(open(meta, encoding='utf-8'))
        pl = m.get('per_language', {})
        summ = ' | '.join(f"{k}: {v.get('macro_f1', 0):.3f}" for k, v in sorted(pl.items()))
        print(f"{stage:<9} seed {seed}: overall {m['val_metrics']['macro_f1']:.4f} | {summ}")


## 8. Big-batch axis (separate from the body ladder)

A larger `--contrastive-batch-size` means **fewer optimizer steps** for the same
pair budget, so it trains *less* at a fixed lr. Compensate by scaling the body lr
**relative to the batch you tuned at** (`--lr-base-batch`, default 8):

- `--lr-scale linear` -> `lr * batch/base` (the linear scaling rule; use for plain SGD)
- `--lr-scale sqrt`   -> `lr * sqrt(batch/base)` (safer with Adam)
- `--lr-scale none`   -> use `--learning-rate` verbatim (default; the ladder uses this)

Keep the **body ladder at batch 8, lr 2e-5** so only the body varies. Use this cell
only to explore the batch/lr tradeoff -- ideally on a *partial* body (`tail2`), which
fits a bigger batch on a 16 GB T4. Watch the `estimated contrastive steps` line each
run prints; scaling lr keeps *per-step* progress similar but step count still drops.


In [ ]:
# Separate from the body ladder: sweep batch and let --lr-scale keep progress comparable.
LR_SCALE    = 'sqrt'      # 'none' | 'linear' | 'sqrt'
BATCHES     = [8, 16, 32] # pairs per step
SWEEP_STAGE = 'tail2'     # partial body -> bigger batch fits a T4; or 'full'

for batch in BATCHES:
    out = f'weights/setfit_s{SEEDS[0]}_{SWEEP_STAGE}_b{batch}'
    cmd = ['python', 'src/run_setfit.py', '--languages', 'EN', 'IT', 'NL',
           '--split-seed', str(SEEDS[0]), '--seed', str(SEEDS[0]),
           '--contrastive-epochs', '1', '--num-iterations', '5',
           '--head-epochs', '20', '--patience', '5',
           '--contrastive-batch-size', str(batch),
           '--lr-scale', LR_SCALE, '--lr-base-batch', '8',
           '--backbone', SETFIT_BACKBONE,
           *STAGES[SWEEP_STAGE], '--out', out]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=os.getcwd())


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/

# A) frozen body (head on fixed pretrained embeddings)
python src/run_setfit.py --head-only --out weights/setfit_s40_headonly

# B) partial body (last 2 encoder blocks + pooling)
python src/run_setfit.py --unfreeze-body-layers 2 --out weights/setfit_s40_tail2

# C) full body (stock SetFit)
python src/run_setfit.py --out weights/setfit_s40_full

# heads on the trained body (one contrastive run, many head A/Bs)
python src/run_setfit.py --head-sweep --body-from weights/setfit_s40_full \
    --sweep-heads torch,mlp,lr:0.1,lr:1,lr:10 --out weights/setfit_s40_full

# bigger contrastive batch -> scale the body lr to keep progress (separate axis)
python src/run_setfit.py --contrastive-batch-size 16 --lr-scale sqrt \
    --lr-base-batch 8 --out weights/setfit_s40_full_b16
```

Outputs per run: `weights/setfit_s<seed>_<stage>/` (model), `..._history.csv`
(per-epoch val macro-F1), `..._run.json` (metadata incl. `body_unfreeze_layers`,
token audit, per-language metrics).
